This notebook showcases the current state of the PDAE API and how it can be used to generate simulated data, train a model and evaluate it. 

It also features checks for extrapolation guarantees of test domains (see Theorem 4.6 of the [preprint](https://arxiv.org/abs/2504.18522)) and MMD tests whether ground truth training and predicted distributions can be distinguished.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../")
sys.path.append("../../")
sys.path.append("../../../")

In [ ]:
import os
import torch
import seaborn as sns

sns.set_theme(style="darkgrid")

from PerturbationExtrapolation.pdae.api import PDAEData, PDAEConfig, PDAEModel

In [ ]:
from PerturbationExtrapolation.pdae.utils import set_random_seed
SEED = 2
set_random_seed(SEED)

In [ ]:
from pathlib import Path
from datetime import datetime

root_path = ""

current_time = datetime.now().strftime('%Y-%m-%d_%H-%M')
pdae_save_dir = Path(root_path) /  current_time
pdae_save_dir.mkdir(parents=True, exist_ok=True)

### Generate simulated data

In [ ]:
import os
import json

from PerturbationExtrapolation.pdae.mixing_functions import NoisyComplexExponential

# dimensionalities
dim_x = 2            # dimension of observations
dim_z_true = 2       # dimension of groundtruth latents
dim_noise_true = 0   # dimension of groundtruth noise variables
# standard deviations of priors
sigma_z = 0.25               # isotropic Gaussian prior for the ground truth latent basal state
sigma_noise_true = 0.5      # isotropic Gaussian prior for groundtruth noise variables

# perturbation settings
n_perturbations = 3  # number of elementary perturbations
perturbation_matrix = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])  # mean shifts of elementary perturbations
# dist = 0.75
# perturbation_matrix = dist * torch.tensor([[1., 1.], [1., 0.], [1., -1.]])  # mean shifts of elementary perturbations

assert perturbation_matrix.shape == (n_perturbations, dim_z_true)
# perturbation_matrix = None  # uncomment to use a randomly initialized perturbation matrix
# perturbation_labels = torch.tensor([[0., 0., 0.], [1., 0., 0.], [0., 1., 0.], [0., 0., 1.], [1., 1., 0.], [0., 0., 2.]])  # labels corresponding to the elementary perturbations
# perturbation_labels = torch.tensor([[0., 0., 0.], [1., 0., 0.], [0., 1., 0.], [0., 0., 1.], [1., 1., 0.], [1., 0., 1.], [0., 1., 1.], [1., 1., 1.]])  # labels corresponding to the elementary perturbations 
# perturbation_labels = torch.tensor([[0., 0., 0.], [1., 0., 0.], [0., 1., 0.], [0., 0., 1.], [1., 1., 0.], [1., 0., 1.], [0., 1., 1.]])  # labels corresponding to the elementary perturbations 
perturbation_labels = torch.tensor([[0., 0., 0.], [1., 0., 0.], [0., 1., 0.], [0., 0., 1.], [1., 1., 0.], [1., 0., 1.]])  # labels corresponding to the elementary perturbations 
assert perturbation_labels.shape[1] == n_perturbations
# perturbation_labels = None  # uncomment to use  all single and double perturbations

# mixing function settings
mixing_scale = 0.5
fixed_mixing = NoisyComplexExponential(sigma_noise=sigma_noise_true, dim_noise=dim_noise_true, scale=mixing_scale)
# fixed_mixing = NoisyIdentity(sigma_noise=sigma_noise_true, dim_noise=dim_noise_true)
# fixed_mixing = None  # uncomment to use a randomly initialized MLP as mixing function
mixing_layer_shapes = [20, 20]  # hidden layer shapes for random true mixing (no. of elements = no. of nonlinearities)

# sample sizes
n_samples = 2**12  # number of observations per dataset
n_samples_heldout = n_samples // 2**3  # number of training samples heldout to compute validation and test losses

# Create a dictionary with all simulation settings
simulation_settings = {
    'seed': SEED,
    'dim_x': dim_x,
    'dim_z_true': dim_z_true,
    'dim_noise_true': dim_noise_true,
    'sigma_z': sigma_z,
    'sigma_noise_true': sigma_noise_true,
    'n_perturbations': n_perturbations,
    'perturbation_matrix': perturbation_matrix.tolist(),
    'perturbation_labels': perturbation_labels.tolist(),
    'fixed_mixing': fixed_mixing.__class__.__name__ if fixed_mixing is not None else None,
    'mixing_scale': mixing_scale,
    'mixing_layer_shapes': mixing_layer_shapes,
    'n_samples': n_samples,
    'n_samples_tr_heldout': n_samples_heldout,
}

# Save the dictionary to a file
with open(pdae_save_dir / 'simulation_settings.json', 'w') as f:
    json.dump(simulation_settings, f, indent=4)

In [ ]:
from PerturbationExtrapolation.pdae.data_generation import get_data
from PerturbationExtrapolation.pdae.utils import show_GT_data

observations, perturbation_labels, perturbation_matrix, true_basal_latents, true_latents, true_mixing = get_data(dim_z_true, dim_noise_true, sigma_noise_true, dim_x, n_perturbations, n_samples, mixing_layer_shapes, sigma_z, perturbation_matrix, perturbation_labels, fixed_mixing)

# split data into train, validation, and test
x_tr = observations[:-2, n_samples_heldout:]
x_tr_heldout = observations[:-2, :n_samples_heldout]
pert_tr = perturbation_labels[:-2]
true_z_tr = true_latents[:-2, n_samples_heldout:]
true_z_tr_heldout = true_latents[:-2, :n_samples_heldout]

x_val = observations[-2]
pert_val = perturbation_labels[-2]
true_z_val = true_latents[-2]

x_te = observations[-1]
pert_te = perturbation_labels[-1]
true_z_te = true_latents[-1]

# ensure that the data is not trainable
assert x_tr.requires_grad is False and pert_tr.requires_grad is False

x_val = x_val.unsqueeze(0)
pert_val = pert_val.unsqueeze(0)
true_z_val = true_z_val.unsqueeze(0)
x_te = x_te.unsqueeze(0)
pert_te = pert_te.unsqueeze(0)
true_z_te = true_z_te.unsqueeze(0)

# show_mixing(true_mixing)
show_GT_data(true_z_tr, true_z_val, true_z_te, x_tr, x_val, x_te, save_dir=pdae_save_dir, ratio_sample=0.2)

In [ ]:
from PerturbationExtrapolation.pdae.dataloader import flatten_observations_perturbations
x_tr_flat, pert_tr_flat = flatten_observations_perturbations(x_tr, pert_tr)
x_tr_heldout_flat, pert_tr_heldout_flat = flatten_observations_perturbations(x_tr_heldout, pert_tr)
z_tr_heldout_flat, _ = flatten_observations_perturbations(true_z_tr_heldout, pert_tr) 
x_val_flat, pert_val_flat = flatten_observations_perturbations(x_val, pert_val)  # add env dimension for pert_val
z_val_flat, _ = flatten_observations_perturbations(true_z_val, pert_val)  # add env dimension for true_z_val
x_te_flat, pert_te_flat = flatten_observations_perturbations(x_te, pert_te)  # add env dimension for pert_te
z_te_flat, _ = flatten_observations_perturbations(true_z_te, pert_te)  # add env dimension for true_z_te
x_tr_flat.shape, pert_tr_flat.shape, x_tr_heldout_flat.shape, pert_tr_heldout_flat.shape, z_tr_heldout_flat.shape, x_val_flat.shape, pert_val_flat.shape, z_val_flat.shape, x_te_flat.shape, pert_te_flat.shape

In [ ]:
observations = torch.cat([x_tr_flat, x_tr_heldout_flat, x_val_flat, x_te_flat], dim=0)
perturbations = torch.cat([pert_tr_flat, pert_tr_heldout_flat, pert_val_flat, pert_te_flat], dim=0)
splits = (
    ["train"] * x_tr_flat.shape[0] +
    ["train_holdout"] * x_tr_heldout_flat.shape[0] +
    ["val"] * x_val_flat.shape[0] +
    ["test"] * x_te_flat.shape[0]
)

### PDAEData

In [ ]:
pdae_data = PDAEData(
    data_name="simulation_noisy_complex_exponential",
    observations=observations,
    perturbations=perturbations,
    splits=splits,
    pert_ctrl=(perturbations==0.0).all(dim=1),
    save_dir=pdae_save_dir
)

In [ ]:
pdae_data.check_extrapolation_guarantees(print_progress=True)

### PDAEConfig

In [ ]:
model_name = 'test_api'
pdae_config = PDAEConfig(**{
    'seed': SEED,
    'model_name': model_name,
    'batch_size': 512,
    'learning_rate': 5e-3,
    'dim_z_model': 2,
    'dim_noise_model': 0,
    'sigma_noise_model': 0.0,
    'decoder_layer_shapes': [64, 64, 64, 64],
    'encoder_layer_shapes': [64, 64, 64, 64],
    'weight_reconstruction_loss': 0.1,
    'weight_perturbation_energy_loss': 1.0,
    'weight_marginal_prior_energy_loss': 0.01,
    'weight_l21': 0.001,
    'update_encoder_on_reconstruction_loss': False,
    'normalize_energy_loss': False,
    'beta': 1.0,
    'use_bias_in_perturbation_matrix': False,
    'use_softplus_after_decoder': False,
    # 'train_split': "train",
    # 'train_holdout_split': "train_holdout",
    # 'val_split': "val",
    # 'test_split': "test",
})

### PDAEModel

In [ ]:
pdae = PDAEModel(device="cuda:0", pdae_data=pdae_data, pdae_config=pdae_config, save_dir=pdae_save_dir)

### Training

In [ ]:
N_EPOCHS = 500

pdae.train(
    num_epochs=N_EPOCHS,
    n_epochs_run_eval=5,
    n_epochs_model_checkpoint=N_EPOCHS,
    n_epochs_plot_losses=50,
    # losses_to_plot=["reconstruction_loss", "perturbation_energy_loss", "val_energy_losses", "val_mean_diffs"],
    n_epochs_plot_distributions=100,
    plot_distribution_ratio_sample=1,
    n_epochs_mmd_test=N_EPOCHS,
    z_tr_holdout=z_tr_heldout_flat,
    z_val=z_val_flat,
    z_te=z_te_flat,
)

### Evaluate the model and benchmark

In [ ]:
from PerturbationExtrapolation.pdae.baselines import get_baselines
from PerturbationExtrapolation.pdae.metrics import get_metrics

pdae.run_eval(
    models=get_baselines(),
    metrics=get_metrics(),
    use_train_holdout=False,
    ratio_subsample_val_test=1,
    run_eval_batched=False,
    return_df=False,
    run_eval_for_train=True
)

In [ ]:
df_evals_agg = pdae.get_agg_eval_results(layout="wide")
df_evals_agg

### Inspect model internals

In [ ]:
pdae.pdae_model.perturbation_matrix.weight

In [ ]:
(pdae.pdae_model.perturbation_matrix.weight[:,0] + pdae.pdae_model.perturbation_matrix.weight[:,1] - pdae.pdae_model.perturbation_matrix.weight[:,2]).abs()

In [ ]:
pdae.pdae_model

### Losses

In [ ]:
pdae.losses.keys()

In [ ]:
pdae.plot_losses()

### MMD test: p-value plots

In [ ]:
pdae.plot_mmd_stats()

### MMD test: distribution plots

In [ ]:
pdae.run_mmd_test(alpha=0.05)

### Visualize ground truth and predicted distributions

In [ ]:
pdae.plot_distributions(
    plot_distribution_ratio_sample=1,
    z_tr_holdout=z_tr_heldout_flat,
    z_val=z_val_flat,
    z_te=z_te_flat,
)

### Load a pretrained model

In [ ]:
LOAD_MODEL = False

if LOAD_MODEL: 
    from pathlib import Path

    model_name = "test_api"
    epoch = 1
    pretrained_model_path = pdae_save_dir / f"pdae_model_{model_name}_epoch_{epoch}.pth"

    pdae_model = PDAEModel(
        pretrained_model_path=pretrained_model_path,
        data_path=pdae_save_dir / "pdae_data_simulation_noisy_complex_exponential.pth",
        config_path=pdae_save_dir / f"pdae_config_{model_name}.json",
        device="cuda:0",
    )

    pdae_model.num_epochs_trained